# PeptideScreen Pipeline v2.0

Complete peptide discovery pipeline: PDB → ProteinMPNN → AF2-Multimer Filter → HADDOCK3 → MD Stability

**Run each cell in order.** Results are saved to Google Drive under `PeptideScreen_Runs/`.

**Prerequisites:**
- Google Colab with GPU runtime (Runtime → Change runtime type → GPU)
- `cns_solve.zip` uploaded to Google Drive (one-time, for HADDOCK3)

In [ ]:
# ============================================================
# CELL 1: Install Dependencies + Clone Pipeline
# ============================================================
# This installs all required packages and clones the pipeline repo.
# Takes ~3-5 minutes on first run.

%cd /content
!rm -rf NewPipeline_07-14
!git clone https://github.com/PMONESKIN/NewPipeline_07-14.git
%cd NewPipeline_07-14

# Core dependencies
!pip install -q pyyaml biopython requests numpy scipy

# Structure + chemistry
!pip install -q rdkit PeptideBuilder meeko gemmi openbabel-wheel

# Docking + MD
!pip install -q haddock3 openmm pdbfixer

# Deep learning
!pip install -q torch

# ColabFold (AF2-Multimer)
!pip install -q "colabfold[alphafold] @ git+https://github.com/sokrypton/ColabFold"

# Verify GPU
import torch
print(f"\nGPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE'}")
print("Setup complete.")

In [ ]:
# ============================================================
# CELL 2: Mount Google Drive + Setup CNS (for HADDOCK3)
# ============================================================
# CNS is the physics engine HADDOCK3 needs.
# Upload cns_solve.zip to Google Drive once (see README).

from google.colab import drive
drive.mount('/content/drive')

import os

# Copy CNS from Drive
cns_zip = '/content/drive/MyDrive/cns_solve.zip'
if os.path.exists(cns_zip):
    !mkdir -p tools
    !cp {cns_zip} tools/
    !cd tools && unzip -q -o cns_solve.zip
    os.environ['CNS_SOLVE'] = '/content/NewPipeline_07-14/tools/cns_solve'
    print("CNS solver ready.")
else:
    print("WARNING: cns_solve.zip not found on Google Drive.")
    print("HADDOCK3 will not work without it.")
    print("Upload cns_solve.zip to the root of your Google Drive.")

# Create results directory on Drive
DRIVE_RESULTS = '/content/drive/MyDrive/PeptideScreen_Runs'
os.makedirs(DRIVE_RESULTS, exist_ok=True)
print(f"Results will be saved to: {DRIVE_RESULTS}")

In [ ]:
# ============================================================
# CELL 3: Configure Target
# ============================================================
# Edit this cell for your target. Everything has defaults.
# For a new target: change pdb_id, receptor_chain, ligand_chain.

config_text = """
project:
  name: PeptideScreen
  version: '2.0'
  description: Computational peptide discovery pipeline.
targets:
- name: Nrf2/KEAP1
  pdb_id: 2FLU
  receptor_chain: X
  ligand_chain: P
  fixed_motif: ETGE
  design_chain_length: 16
  mpnn_temperatures: [0.1, 0.2, 0.3]
docking:
  methods: [haddock3]
  haddock3:
    max_candidates: 20
    sampling:
      rigidbody: 1000
      rigidbody_fast: 200
      flexref: 200
      emref: 200
md_stability:
  force_field: charmm36
  temperature_K: 310.15
  ionic_strength_molar: 0.15
  default_ns: 50.0
  checkpoint_interval_ps: 500
  report_interval_ps: 50
candidates:
  initial_pool: 50
  final_shortlist: 10
  length:
    min: 5
    max: 30
  filters:
    max_molecular_weight: 2500
    flag_aggregation_prone: true
outputs:
  structures: data/structures/
  candidates: data/candidates/
  docking: data/docking/
  processed: data/processed/
  reports: outputs/reports/
  figures: outputs/figures/
"""

with open('config.yaml', 'w') as f:
    f.write(config_text)
print("Config saved.")

In [ ]:
# ============================================================
# CELL 4: Keep Colab Alive
# ============================================================
%%javascript
function ClickConnect(){
  console.log("Keeping alive");
  document.querySelector("colab-connect-button").click()
}
setInterval(ClickConnect, 60000)

In [ ]:
# ============================================================
# CELL 5: Setup ProteinMPNN (one-time)
# ============================================================
!python -u modules/02_design/setup_proteinmpnn.py

In [ ]:
# ============================================================
# CELL 6: PHASE 0 — Fetch PDB + Analyze Interface
# ============================================================
import sys, importlib.util
from pathlib import Path
ROOT = Path('/content/NewPipeline_07-14')
sys.path.insert(0, str(ROOT))
from modules.run_manager import RunManager

# Create new run
rm = RunManager()
RUN_DIR = str(rm.run_dir)
print(f"Run directory: {RUN_DIR}")

# Save RUN_DIR for subsequent cells
with open('/content/current_run.txt', 'w') as f:
    f.write(RUN_DIR)

def load_mod(path):
    p = ROOT / path
    spec = importlib.util.spec_from_file_location(p.stem, str(p))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

print("\n--- Fetching PDB ---")
load_mod('modules/01_targets/fetch_structures.py').run(run_dir=RUN_DIR)

print("\n--- Analyzing Interface ---")
load_mod('modules/01_targets/analyze_interface.py').run(run_dir=RUN_DIR)

print("\nPhase 0 complete.")

In [ ]:
# ============================================================
# CELL 7: PHASE 1 — Backbone Extraction
# ============================================================
# Extracts the binding region backbone and energy minimizes it.
# Change PEPTIDE_LENGTH to design shorter/longer peptides.

PEPTIDE_LENGTH = 16  # Change this for different lengths (e.g., 8)

with open('/content/current_run.txt') as f:
    RUN_DIR = f.read().strip()

print(f"Extracting {PEPTIDE_LENGTH}-residue backbone...")
load_mod('modules/02_design/backbone_extract.py').run(
    run_dir=RUN_DIR, length=PEPTIDE_LENGTH
)
print("Phase 1 complete.")

In [ ]:
# ============================================================
# CELL 8: PHASE 2 — ProteinMPNN Sequence Design
# ============================================================
# Designs peptide sequences that fit the binding backbone.
# ETGE motif is locked if specified in config.

with open('/content/current_run.txt') as f:
    RUN_DIR = f.read().strip()

print("--- ProteinMPNN Design ---")
load_mod('modules/02_design/design_peptides.py').run(run_dir=RUN_DIR)

print("\n--- Building Candidate Pool ---")
load_mod('modules/02_design/candidate_pool.py').run(run_dir=RUN_DIR)

# Show candidates
import json
from modules.run_manager import RunManager
rm = RunManager(run_dir=RUN_DIR)
candidates = rm.load_candidates()
print(f"\n{len(candidates)} unique candidates designed.")
print("\nFirst 5:")
for c in candidates[:5]:
    print(f"  {c['id']}  {c['sequence']}  MPNN={c.get('mpnn_score', 'N/A')}")

print("\nPhase 2 complete.")

In [ ]:
# ============================================================
# CELL 9: PHASE 3 — AF2-Multimer Filter
# ============================================================
# Predicts peptide-receptor complex with AlphaFold2-Multimer.
# Filters by: ipTM > 0.7, pLDDT > 80, interface PAE < 7.0
# ~1-2 minutes per candidate on GPU.
#
# Adjust thresholds if too few/many candidates pass.

IPTM_CUTOFF = 0.7    # Lower = more permissive
PLDDT_CUTOFF = 80.0   # Lower = more permissive
PAE_CUTOFF = 7.0      # Higher = more permissive

with open('/content/current_run.txt') as f:
    RUN_DIR = f.read().strip()

print("Running AF2-Multimer on all candidates...")
print(f"Thresholds: ipTM>{IPTM_CUTOFF}, pLDDT>{PLDDT_CUTOFF}, PAE<{PAE_CUTOFF}")

load_mod('modules/03_docking/af2_filter.py').run(
    run_dir=RUN_DIR,
    iptm_cutoff=IPTM_CUTOFF,
    plddt_cutoff=PLDDT_CUTOFF,
    pae_cutoff=PAE_CUTOFF,
)

# Show results
from modules.run_manager import RunManager
rm = RunManager(run_dir=RUN_DIR)
candidates = rm.load_candidates()
passed = [c for c in candidates if c.get('af2_pass')]
print(f"\n{len(passed)}/{len(candidates)} candidates passed AF2 filter.")

print("\nPhase 3 complete.")

In [ ]:
# ============================================================
# CELL 10: PHASE 4 — HADDOCK3 Full Docking
# ============================================================
# Runs full HADDOCK3 (rigidbody + flexref + emref) on
# AF2-validated candidates only. Uses AF2-predicted structures.
# ~30-50 min per candidate.

MAX_CANDIDATES = None  # None = all AF2-passed, or set a number

with open('/content/current_run.txt') as f:
    RUN_DIR = f.read().strip()

print("Running HADDOCK3 full docking on AF2-validated candidates...")
load_mod('modules/03_docking/haddock3_dock.py').run(
    run_dir=RUN_DIR, mode='full', n=MAX_CANDIDATES
)

print("\n--- Ranking ---")
load_mod('modules/03_docking/score_ranking.py').run(
    run_dir=RUN_DIR, mode='full'
)

print("\nPhase 4 complete.")

In [ ]:
# ============================================================
# CELL 11: PHASE 5 — Modification (Interactive)
# ============================================================
# Add CPP tags (rrrrrrrr), D-amino acids, truncate, etc.
# Then re-dock modified peptides through HADDOCK3.
#
# Type candidate IDs to modify, 'done' when finished.

with open('/content/current_run.txt') as f:
    RUN_DIR = f.read().strip()

modify = load_mod('modules/04_modification/modify_peptides.py')
modified = modify.run(run_dir=RUN_DIR)

if modified:
    print("\n--- Re-docking modified candidates ---")
    redock = load_mod('modules/04_modification/redock_modified.py')
    redock.run(run_dir=RUN_DIR, mode='full')

print("\nPhase 5 complete.")

In [ ]:
# ============================================================
# CELL 12: PHASE 6 — Property Prediction
# ============================================================
# Computes physicochemical properties, protease cleavage sites,
# and permeability estimates for all candidates.

with open('/content/current_run.txt') as f:
    RUN_DIR = f.read().strip()

print("--- Physicochemical ---")
load_mod('modules/05_properties/physicochemical.py').run(run_dir=RUN_DIR)

print("\n--- Protease Stability ---")
load_mod('modules/05_properties/protease_stability.py').run(run_dir=RUN_DIR)

print("\n--- Permeability ---")
load_mod('modules/05_properties/permeability.py').run(run_dir=RUN_DIR)

print("\n--- Properties Report ---")
load_mod('modules/05_properties/properties_report.py').run(run_dir=RUN_DIR)

print("\nPhase 6 complete.")

In [ ]:
# ============================================================
# CELL 13: PHASE 7 — MD Stability (OpenMM)
# ============================================================
# Runs molecular dynamics on top candidates to confirm
# binding stability. 50ns per candidate, ~13 hours each on T4.

MD_TOP_N = 3      # How many candidates to simulate
MD_NS = 50.0      # Simulation length in nanoseconds

with open('/content/current_run.txt') as f:
    RUN_DIR = f.read().strip()

print(f"Running MD stability on top {MD_TOP_N} candidates ({MD_NS} ns each)...")
print(f"Estimated time: {MD_TOP_N * 13:.0f} hours on T4 GPU")

try:
    load_mod('modules/03_docking/md_stability.py').run(
        run_dir=RUN_DIR, n=MD_TOP_N, ns=MD_NS
    )
    print("\nPhase 7 complete.")
except Exception as e:
    print(f"MD failed: {e}")
    print("Try running candidates individually with the OPENMM_V2 notebook.")

In [ ]:
# ============================================================
# CELL 14: PHASE 8 — Final Report + Save to Drive
# ============================================================
# Generates the final ranked report and copies everything
# to Google Drive for permanent storage.

import shutil
from pathlib import Path

with open('/content/current_run.txt') as f:
    RUN_DIR = f.read().strip()

print("--- Generating Final Report ---")
load_mod('modules/03_docking/docking_report.py').run(run_dir=RUN_DIR)

# Copy entire run to Google Drive
run_name = Path(RUN_DIR).name
drive_dest = f'/content/drive/MyDrive/PeptideScreen_Runs/{run_name}'

print(f"\n--- Saving to Google Drive ---")
print(f"  From: {RUN_DIR}")
print(f"  To:   {drive_dest}")

if os.path.exists(drive_dest):
    shutil.rmtree(drive_dest)
shutil.copytree(RUN_DIR, drive_dest)

print(f"\n{'='*60}")
print(f"  PIPELINE COMPLETE")
print(f"{'='*60}")
print(f"  Run: {run_name}")
print(f"  Google Drive: PeptideScreen_Runs/{run_name}/")
print(f"  Reports:")
print(f"    reports/docking_report.md")
print(f"    reports/properties_report.md")
print(f"{'='*60}")

In [ ]:
# ============================================================
# CELL 15: View Results
# ============================================================
# Quick summary of all candidates and their scores.

import json
from modules.run_manager import RunManager

with open('/content/current_run.txt') as f:
    RUN_DIR = f.read().strip()

rm = RunManager(run_dir=RUN_DIR)
candidates = rm.load_candidates()

# Sort by best available score
for field in ['haddock_full_score', 'haddock_fast_score']:
    scored = [c for c in candidates if c.get(field) is not None]
    if scored:
        scored.sort(key=lambda c: c[field])
        break

print(f"{'Rank':<5} {'ID':<16} {'Sequence':<22} {'HADDOCK':<10} {'ipTM':<8} {'pLDDT':<8} {'Stable?':<8} {'Protease'}")
print(f"{'-'*5} {'-'*16} {'-'*22} {'-'*10} {'-'*8} {'-'*8} {'-'*8} {'-'*8}")

for rank, c in enumerate(scored[:20] if scored else candidates[:20], 1):
    seq = c['sequence'][:20] + '..' if len(c['sequence']) > 20 else c['sequence']
    hadd = f"{c.get('haddock_full_score', c.get('haddock_fast_score', 'N/A'))}"
    if isinstance(hadd, float): hadd = f"{hadd:.1f}"
    iptm = f"{c.get('af2_iptm', 'N/A')}"
    if isinstance(iptm, float): iptm = f"{iptm:.3f}"
    plddt = f"{c.get('af2_plddt', 'N/A')}"
    if isinstance(plddt, float): plddt = f"{plddt:.1f}"
    pc = c.get('physicochemical', {})
    stable = 'yes' if pc and not pc.get('is_unstable') else ('NO' if pc else '?')
    prot = c.get('protease_cleavage', {}).get('protease_risk', '?')
    print(f"{rank:<5} {c['id']:<16} {seq:<22} {hadd:<10} {iptm:<8} {plddt:<8} {stable:<8} {prot}")